In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mkechinov/ecommerce-events-history-in-electronics-store")
print("Path to dataset files:", path)

file_path = path + "/events.csv"

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

data = pd.read_csv(file_path)

# Primary category
data["primary_category"] = data["category_code"].str.split(".").str[0]

# Keep only funnel events
funnel_event = data[data["event_type"].isin(["view", "cart", "purchase"])].copy()

# Stage flags
funnel_event["View"] = funnel_event["event_type"] == "view"
funnel_event["Cart"] = funnel_event["event_type"] == "cart"
funnel_event["Purchase"] = funnel_event["event_type"] == "purchase"

# Fill Unclassified
funnel_event_cleaned = funnel_event.copy()
funnel_event_cleaned["primary_category"] = funnel_event_cleaned["primary_category"].fillna("Unclassified")

# Max stage reached by each user within each category
user_category_stages = (
    funnel_event_cleaned
    .groupby(["user_id", "primary_category","price"])[["View", "Cart", "Purchase"]]
    .max()
    .reset_index()
)
# Convert booleans to ints so sums are counts
user_category_stages[["View", "Cart", "Purchase"]] = user_category_stages[["View", "Cart", "Purchase"]].astype(int)

total_views = user_category_stages["View"].sum()

# Category-level funnel counts
category_funnel_summary = (
    user_category_stages
    .groupby("primary_category")[["View", "Cart", "Purchase"]]
    .sum()
    .sort_values(by="View", ascending=False)
)

# Safe divide (no def; just replace 0 with NaN)
category_funnel_summary[["View", "Cart", "Purchase"]] = category_funnel_summary[["View", "Cart", "Purchase"]].replace(0, np.nan)

# Conversion rates (as %)
category_funnel_summary["view_to_cart"] = category_funnel_summary["Cart"] / category_funnel_summary["View"] * 100
category_funnel_summary["cart_to_purchase"] = category_funnel_summary["Purchase"] / category_funnel_summary["Cart"] * 100
category_funnel_summary["view_to_purchase"] = category_funnel_summary["Purchase"] / category_funnel_summary["View"] * 100

category_funnel_summary["percentage_of_total_views"] = category_funnel_summary["View"] / total_views * 100

# Calculate total revenue from views per category
category_funnel_summary["Potential Revenue (View-based)"] = (
    user_category_stages
    .groupby("primary_category")
    .apply(lambda x: (x["price"] * x["View"]).sum())
)

# Calculate total revenue from purchases per category
category_funnel_summary["Realized Revenue (Purchased-based)"] = (
    user_category_stages
    .groupby("primary_category")
    .apply(lambda x: (x["price"] * x["Purchase"]).sum())
)

df = category_funnel_summary.reset_index()  # bring primary_category back as column

# Group small categories into "Other"
threshold = 1.0
df["primary_category"] = df.apply(
    lambda x: x["primary_category"] if x["percentage_of_total_views"] >= threshold else "Other",
    axis=1,
)

# IMPORTANT: aggregate COUNTS first, then recompute rates
df = df.groupby("primary_category", as_index=False)[["View", "Cart", "Purchase"]].sum()

# Recompute share + conversion AFTER grouping
df[["View", "Cart", "Purchase"]] = df[["View", "Cart", "Purchase"]].replace(0, np.nan)

df["percentage_of_total_views"] = df["View"] / df["View"].sum() * 100
df["view_to_cart"] = df["Cart"] / df["View"] * 100
df["cart_to_purchase"] = df["Purchase"] / df["Cart"] * 100
df["view_to_purchase"] = df["Purchase"] / df["View"] * 100

# -----------------------------
# Pie Chart
# -----------------------------
plt.figure(figsize=(8, 8))
plt.pie(
    df["percentage_of_total_views"],
    labels=df["primary_category"],
    autopct="%1.1f%%",
    startangle=90,
)
plt.title("Share of Total Views by Category")
plt.tight_layout()
plt.show()

# -----------------------------
# Bar Chart: View -> Purchase Conversion
# -----------------------------
df_conversion = df.sort_values("view_to_purchase", ascending=False)

plt.figure(figsize=(10, 5))
plt.bar(df_conversion["primary_category"], df_conversion["view_to_purchase"])
plt.xticks(rotation=75, ha="right")
plt.title("View → Purchase Conversion by Category")
plt.ylabel("View to Purchase Conversion (%)")
plt.tight_layout()
plt.show()